# Live exercise: from notebook code to a pipeline

A different question this time: **which device do I skip most on?**

Below is the answer the notebook way: correct, top to bottom, no functions.
Your job is to rebuild it as a pipeline, one function at a time, checking
each one before you build the next on top of it. Then you move it into a
script and run it from two different folders.

The rules from notebook 01:

* one function per stage, each with a one-line docstring saying what it
  returns
* only `main()` prints; every other function returns a value or writes a
  file
* uncomment the `assert` lines once your function exists. They are the real
  numbers, so they tell you when you are right

In [ ]:
import os
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

here = Path.cwd()
while not (here / "data" / "messy" / "plays_messy.csv").exists() and here != here.parent:
    here = here.parent
os.chdir(here)

MESSY = Path("data/messy/plays_messy.csv")
OUTPUT = Path("output")

## The notebook way

Run it, read it, and notice where one stage ends and the next begins.
Those boundaries are where your functions go.

In [ ]:
raw = pd.read_csv(MESSY, keep_default_na=False)

df = raw.rename(columns={"minutes played ": "minutes_played"})
df = df.drop_duplicates().copy()
df["genre"] = df["genre"].str.strip().str.title().replace("", pd.NA)
df["device"] = df["device"].str.strip().str.lower().replace("", "unknown")
df["country"] = df["country"].str.strip().replace(["NA", ""], pd.NA)
for column in ["genre", "country"]:
    lookup = df.dropna(subset=[column]).groupby("artist_name")[column].first()
    df[column] = df[column].fillna(df["artist_name"].map(lookup))
dates = pd.to_datetime(df["played_at"], format="%Y-%m-%d", errors="coerce")
dates = dates.fillna(pd.to_datetime(df["played_at"], format="%d/%m/%Y", errors="coerce"))
dates = dates.fillna(pd.to_datetime(df["played_at"], format="%d-%m-%Y", errors="coerce"))
df["played_at"] = dates.dt.strftime("%Y-%m-%d")
minutes = df["minutes_played"].str.replace("min", "").str.replace(" ", "")
df["minutes_played"] = pd.to_numeric(minutes, errors="coerce")

rates = df.groupby("device").agg(plays=("play_id", "count"), skipped=("skipped", "sum"))
rates = rates.reset_index()
rates["skip_rate"] = (rates["skipped"] / rates["plays"] * 100).round(1)
rates = rates.sort_values("skip_rate", ascending=False).reset_index(drop=True)
rates

## 1. `load_messy(path)`

Returns the file as a DataFrame, changing nothing.

In [ ]:
# TODO: def load_messy(path):


# raw = load_messy(MESSY)
# assert raw.shape == (2223, 9)

## 2. `clean_plays(df)`

Returns the cleaned plays. This is your session 9 function: copy it (with
its helpers) from notebook 01 or from
`sessions/session-09/solutions/cleaning.py`, or use your own if it gives
the same result.

In [ ]:
# TODO: def clean_plays(df):


# plays = clean_plays(raw)
# assert len(plays) == 2183
# assert (plays["device"] == "unknown").sum() == 69

## 3. `skip_rate_by_device(plays)`

Returns one row per device, with the columns `device`, `plays`, `skipped`
and `skip_rate` (a percentage, one decimal), highest rate first.

In [ ]:
# TODO: def skip_rate_by_device(plays):


# rates = skip_rate_by_device(plays)
# assert len(rates) == 6
# assert rates["plays"].sum() == 2183
# assert rates["skipped"].sum() == 258
# rates

**Think about it:** look at the top row of your table. Is it a device?
Where did it come from, and should it be in the chart?

*Your answer here.*

## 4. `save_csv(table, path)` and `save_chart(rates, path)`

`save_csv` writes the table, making the folder if it is missing.
`save_chart` draws a bar chart of the skip rate per device, with the answer
as its title, labels like `14.1%`, and saves it as a PNG. Neither prints.

In [ ]:
# TODO: def save_csv(table, path):


# TODO: def save_chart(rates, path):


# save_csv(rates, OUTPUT / "device_skip_rate.csv")
# save_chart(rates, OUTPUT / "device_skip_rate.png")

In [ ]:
# from IPython.display import Image
# Image(filename="output/device_skip_rate.png", width=640)

## 5. `check_rates(path, plays)`

Reads the CSV back and stops with an `AssertionError` if the plays or the
skips in the file do not add up to the ones in `plays`.

In [ ]:
# TODO: def check_rates(path, plays):


# check_rates(OUTPUT / "device_skip_rate.csv", plays)

## 6. `main()`

Calls everything in order, and is the only function that prints. Print a
line after each stage, as it happens: how many rows were loaded, how many
are left after cleaning, the table, each file as it is written, and that
the check passed. If a run stops halfway, the last line says where.

In [ ]:
# TODO: def main():


# main()

## 7. Move it into a script

Open `sessions/session-10/exercises/device_pipeline.py`. It holds the same
notebook-way code as above.

1. Run it from the repository root in a terminal. It works.
2. `cd sessions/session-10` and run `python exercises/device_pipeline.py`.
   Read the error.
3. Replace the notebook-way code with your functions, add
   `if __name__ == "__main__":` at the bottom, and build the paths from
   `__file__` with `pathlib`, so it works from any folder.

Then run this cell. It runs your script from three folders, and all three
should print the same thing.

In [ ]:
# import subprocess
# import sys
#
# script = Path("sessions/session-10/exercises/device_pipeline.py").resolve()
# for folder in [".", "sessions/session-10", "data/clean"]:
#     result = subprocess.run([sys.executable, str(script)], cwd=folder,
#                             capture_output=True, text=True)
#     print(f"--- from {folder}")
#     print(result.stdout or result.stderr[-300:])

## If you have time

Import your script in the cell below with `sys.path.insert` and `import`,
the way notebook 01 imported the genre pipeline. Does anything print? Use
its `skip_rate_by_device` on just the 2025 plays.

In [ ]:
# TODO